# Assignment 1

![](https://media.giphy.com/media/xT9C25UNTwfZuk85WP/giphy-downsized-large.gif)

Remember the rules of ~Fight~ Code Club:
1. ALWAYS DOCUMENT
2. Cite resources that you use (paste links)
3. Include the names people who you worked with
4. Be neat and organized

In [38]:
#NOTE!!!: Install required libraries if not already installed so the codes can runn
!pip install pandas numpy requests geopandas

## Scrape and Clean data

Based on you proposal, scrape or collect your data:

1. One variable must be either: (40 pts)
    1. scraped from the web OR;
    2. collected from an API AND you must create one new variable that is "new" to the best of your knowledge (combination of other variables representing something new).

2. You must have at least 3 variables, but you may include as many as you want into you final dataset. Likely, you will want to include more to make graphs and regressions. (30pts)

3. You must *be able* to run a regression that makes some sense with this data (the regression doesn't have to be a complete model). Briefly describe one regression you would run with your variables. (**DO NOT** run a regression, yet). (15pts)

4. You must have one combined and cleaned dataset (15 pts)

You must submit one python notebook on how you scraped/gathered data from an api, and how you combined and cleaned you data. I should be able to run your code and reproduce your final data set.  

The other variables that you choose to include do not have to be collected by API or webscraped, but you do have to combine the files and clean the dataset with python.

Thus, you must submit:
- Your finalized data set (only one) (note: you may add more variables in the future).
- Your documented python notebook
- Any associated data files needed to produce the final dataset.

You will be evaluated on:
- Completeness of the data
- Quality of the code 
- The creativity of the new variable/webscraped data you gather

Be sure to upload ALL associated files for your code to run. I will run your code from beginning to end - make it easy for me to replicate your code.

# Project: Housing Affordability in Greater Boston

## Research Question
Does zoning regulation, specifically restrictions on building height 
and multifamily housing , affect housing affordability in Greater Boston?

## Planned Regression
log(median_rent) = β0 + β1·mf_share + β2·log(median_hh_income) + β3·mean_max_height_ft + ε

If zoning allows taller/denser buildings, rents should be lower after controlling for income and existing density..

## Resources Used
- Census ACS API: https://api.census.gov/data/2022/acs/acs5
- Boston Zoning: https://data.boston.gov/dataset/boston-zoning-subdistricts
- TIGER Shapefiles: https://www2.census.gov/geo/tiger/TIGER2022/TRACT/
- HUD Rent Burden: https://www.hud.gov/program_offices/comm_planning/affordability

In [39]:
# Imported data
import warnings
warnings.filterwarnings('ignore')
import requests, os
import pandas as pd
import numpy as np
import geopandas as gpd

CENSUS_KEY = "449daedeb4203d7011951c788e288009b73360d8"
ACS_BASE   = "https://api.census.gov/data/2022/acs/acs5"

print("Imported - OK")

Imported - OK


In [40]:
# Housing and income data from Census ACS 5-Year 2022 API
# Source: https://api.census.gov/data/2022/acs/acs5

ACS_VARS = {
    "B25001_001E": "total_units",
    "B25024_004E": "units_2",
    "B25024_005E": "units_3_4",
    "B25024_006E": "units_5_9",
    "B25024_007E": "units_10_19",
    "B25024_008E": "units_20plus",
    "B25064_001E": "median_rent",
    "B25077_001E": "median_value",
    "B25003_003E": "renter_occupied",
    "B25003_001E": "total_occupied",
    "B01003_001E": "population",
    "B19013_001E": "median_hh_income",
}

r = requests.get(ACS_BASE, params={
    "get": f"NAME,{','.join(ACS_VARS.keys())}",
    "for": "tract:*",
    "in":  "state:25",       # Massachusetts
    "key": CENSUS_KEY,
}, timeout=60)
r.raise_for_status()

raw = r.json()
acs = pd.DataFrame(raw[1:], columns=raw[0])
acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]
acs = acs.rename(columns=ACS_VARS).drop(columns=["state","county","tract","NAME"], errors="ignore")

# I convert the data to numbers and replace the Census placeholder values with NaN
acs[list(ACS_VARS.values())] = acs[list(ACS_VARS.values())].apply(pd.to_numeric, errors="coerce")
acs = acs.replace(-666666666, np.nan).replace(-999999999, np.nan)

print(f"Tracts pulled: {len(acs):,}")
acs.head(3)

Tracts pulled: 1,620


,total_units,units_2,units_3_4,units_5_9,units_10_19,units_20plus,median_rent,median_value,renter_occupied,total_occupied,population,median_hh_income,GEOID
0,4905,640,1137,499,176,116,1106.0,772200.0,471,1996,3630,91447.0,25001010100
1,4862,172,16,6,0,0,1289.0,683300.0,279,1975,4352,97396.0,25001010206
2,3449,21,29,22,17,17,NaN,701300.0,169,1031,1627,89243.0,25001010208


In [41]:
# NEW VARIABLE: Affordability Stress Index (ASI)
# = median_rent / (median_hh_income / 12)
# Share of monthly income spent on rent
# Not published by Census — was built by combining B25064 + B19013
# HUD defines rent-burdened as ASI > 0.30 (30%))
# Source: https://www.hud.gov/program_offices/comm_planning/affordability

acs["affordability_stress_index"] = acs["median_rent"] / (acs["median_hh_income"] / 12)
acs["rent_burdened"] = (acs["affordability_stress_index"] > 0.30).astype(float)
acs.loc[acs["affordability_stress_index"].isna(), "rent_burdened"] = np.nan

acs["mf_share"]   = (acs["units_2"] + acs["units_3_4"] + acs["units_5_9"] +
                     acs["units_10_19"] + acs["units_20plus"]) / acs["total_units"].replace(0, np.nan)
acs["log_rent"]   = np.log(acs["median_rent"].replace(0, np.nan))
acs["log_income"] = np.log(acs["median_hh_income"].replace(0, np.nan))

print(f"Median ASI: {acs['affordability_stress_index'].median():.1%}")
print(f"Rent-burdened tracts: {acs['rent_burdened'].mean():.1%}")
acs[["GEOID","median_rent","median_hh_income","affordability_stress_index","mf_share"]].head(3)

Median ASI: 20.6%
Rent-burdened tracts: 14.3%


,GEOID,median_rent,median_hh_income,affordability_stress_index,mf_share
0,25001010100,1106.0,91447.0,0.145133,0.523547
1,25001010206,1289.0,97396.0,0.158816,0.039901
2,25001010208,NaN,89243.0,NaN,0.030734


In [42]:
# Scraped Boston Zoning Subdistricts GeoJSON — Open Data
# Source: https://data.boston.gov/dataset/boston-zoning-subdistricts

URL = (
    "https://data.boston.gov/dataset/61396e0c-f1e6-4625-ab38-08e9e7e355bd/"
    "resource/1388010c-16fb-419d-888b-2cae355dc3a8/download/boston_zoning_subdistricts.geojson"
)
FALLBACK = "https://bostonopendata-boston.opendata.arcgis.com/datasets/boston::zoning-subdistricts.geojson"

try:
    resp = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=45)
    resp.raise_for_status()
    zoning = gpd.GeoDataFrame.from_features(resp.json()["features"], crs="EPSG:4326")
except:
    resp = requests.get(FALLBACK, headers={"User-Agent": "Mozilla/5.0"}, timeout=45)
    zoning = gpd.GeoDataFrame.from_features(resp.json()["features"], crs="EPSG:4326")

# Standardize column names
for old, new in [("Max_Height","max_height_ft"), ("Max_FAR","max_far"),
                  ("Max_Dwelling_Units_Per_Area","max_du_per_acre"),
                  ("Zoning_District","zoning_district")]:
    if old in zoning.columns:
        zoning = zoning.rename(columns={old: new})

for col in ["max_height_ft","max_far","max_du_per_acre"]:
    if col in zoning.columns:
        zoning[col] = pd.to_numeric(zoning[col], errors="coerce")

print(f"Zoning zones scraped: {len(zoning):,}")
zoning[["zoning_district","max_height_ft","max_far"]].head(3)

Zoning zones scraped: 1,640


,zoning_district,max_height_ft,max_far
0,Mission Hill Neighborhood,55.0,3.0
1,South Boston Neighborhood,65.0,2.0
2,South Boston,NaN,4.0


In [43]:
# I’ve rolled up the zoning data to the tract level using a spatial join.
# Tract shapefiles: https://www2.census.gov/geo/tiger/TIGER2022/TRACT/

try:
    tracts = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_25_tract.zip")
    tracts = tracts[tracts["COUNTYFP"] == "025"]  # Suffolk County (Boston)

    tracts_p = tracts.to_crs("EPSG:26986")
    zoning_p = zoning.to_crs("EPSG:26986")

    z_cols = [c for c in ["max_height_ft","max_far","max_du_per_acre","geometry"] if c in zoning_p.columns]
    joined = gpd.sjoin(zoning_p[z_cols], tracts_p[["GEOID","geometry"]],
                       how="left", predicate="intersects")

    agg = [c for c in ["max_height_ft","max_far","max_du_per_acre"] if c in joined.columns]
    zoning_by_tract = joined.groupby("GEOID")[agg].mean().reset_index()
    zoning_by_tract.columns = ["GEOID"] + [f"mean_{c}" for c in agg]
    print(f"Zoning aggregated: {len(zoning_by_tract):,} tracts")

except Exception as e:
    print(f"Spatial join failed: {e} — zoning columns will be NaN")
    zoning_by_tract = pd.DataFrame({"GEOID": acs["GEOID"],
                                     "mean_max_height_ft": np.nan,
                                     "mean_max_far": np.nan})

Zoning aggregated: 213 tracts


In [44]:
# Merged ACS + zoning into one  dataset

final = acs.merge(zoning_by_tract, on="GEOID", how="left")
final = final[final["population"].fillna(0) > 0]     # drop uninhabited tracts
final = final.drop_duplicates(subset="GEOID")
final = final.sort_values("GEOID").reset_index(drop=True)

print(f"Final dataset: {final.shape[0]:,} tracts x {final.shape[1]} columns")
print(f"Tracts with zoning data: {final['mean_max_height_ft'].notna().sum():,}")
final.head()

Final dataset: 1,602 tracts x 21 columns
Tracts with zoning data: 202


,total_units,units_2,units_3_4,units_5_9,units_10_19,units_20plus,median_rent,median_value,renter_occupied,total_occupied,...,median_hh_income,GEOID,affordability_stress_index,rent_burdened,mf_share,log_rent,log_income,mean_max_height_ft,mean_max_far,mean_max_du_per_acre
0,4905,640,1137,499,176,116,1106.0,772200.0,471,1996,...,91447.0,25001010100,0.145133,0.0,0.523547,7.008505,11.423515,NaN,NaN,NaN
1,4862,172,16,6,0,0,1289.0,683300.0,279,1975,...,97396.0,25001010206,0.158816,0.0,0.039901,7.161622,11.486540,NaN,NaN,NaN
2,3449,21,29,22,17,17,NaN,701300.0,169,1031,...,89243.0,25001010208,NaN,NaN,0.030734,NaN,11.399118,NaN,NaN,NaN
3,2935,28,25,0,0,0,NaN,646900.0,57,1363,...,68633.0,25001010304,NaN,NaN,0.018058,NaN,11.136529,NaN,NaN,NaN
4,3458,118,57,0,0,0,1683.0,609100.0,128,1361,...,81078.0,25001010306,0.249093,0.0,0.050607,7.428333,11.303167,NaN,NaN,NaN


In [45]:
# Saved final dataset
final.to_csv("housing_affordability_dataset.csv", index=False)
print("Saved: housing_affordability_dataset.csv")
print(f"Shape: {final.shape}")

Saved: housing_affordability_dataset.csv
Shape: (1602, 21)


## Note!!!!!! on Future Data (Work in Progress)

I am also experimenting with parcel-level data from MassGIS Assessor Data.
Source: https://www.mass.gov/info-details/massgis-data-property-tax-parcels

This will be beneficial because:
- The current dataset is at the census tract level 
- Parcel data gives individual property-level observations (which is much more precise)
- It would allow me to match actual assessed values and property types 
  directly to zoning boundaries rather than averaging across a whole tract
- This is important for my final spatial RDD, which needs observations 
  close to zoning boundary lines, tracts are too big for that

Still experimenting with how to clean and join it, so it is not included 
in this submission but will most likely be added in a future version of the dataset.

School district boundaries (NCES SABS 2019-20) — to control for 
  school quality near zoning boundaries,this will be included
- Water and road features (TIGER REST API) — I’ll use them as natural boundary controls for the spatial RDD.
- MAPC Zoning Atlas (ArcGIS REST API) — to get detailed, parcel-level zoning data for all 86 municipalities in Greater Boston.

**These additions will allow for a more complete model and are necessary for the full spatial RDD design in the final project.**


prof k 
?you need to document your webscraping in more detail; I can tell where you wrote it yourself (API) and where you got help from chatgpt (webscraping). 


No reason to use try/except for rolling up data into tracts.  I need to see a check and evidence that the tracts are correctly matching the shape files.

Does boston/MA have open data like nyc? You might be able to find the permit/valuations information at their tax office (like in nyc).

Aren't you glad you are doing a project that applies your skills?

extra points for creativity of the project
grade: 88